# LMSYS Chatbot Arena — DeBERTa v3
End-to-end pipeline: load → preprocess → train → submit

This was a DeBERTa trained model. Was the third version due to training issues where all values were becoming NaNs during training. 

Explanation of errors faced on this approach:
NaN predictions during validation, the Trainer requires the label column to be named labels, not label. Fixed by renaming before dataset creation.
CUDA error — notebook was set to the wrong accelerator. Fixed by switching to T4 x2.
NaN predictions again — caused by load_best_model_at_end=True. With T4 x2 (DDP), the Trainer saves the checkpoint from rank-0 only and then reloads it, but the reload doesn't correctly redistribute weights across both GPUs. The model ends up partially uninitialised → NaN logits. Fixed by reverting to save_strategy="no".


Uses Kaggle T4x2 GPU-s (NVIDIA TESLA)

Public log loss score: 1.09868

## 1. Setup

In [1]:
import os

# ── Force single GPU — prevents multi-GPU loss gathering issues on T4 x2 ──────
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ── PATH SWITCHER ─────────────────────────────────────────────────────────────
# Auto-detects environment. Override manually if needed.
if os.path.exists("/kaggle/input"):
    ENV = "kaggle"
else:
    ENV = "local"

# ── Manual override (uncomment one line to force) ─────────────────────────────
# ENV = "local"
# ENV = "kaggle"
# ─────────────────────────────────────────────────────────────────────────────

if ENV == "local":
    DATA_DIR   = "Data/"
    OUTPUT_DIR = "."
    MODEL_NAME = "distilbert-base-uncased"
elif ENV == "kaggle":
    DATA_DIR   = "/kaggle/input/competitions/lmsys-chatbot-arena/"
    OUTPUT_DIR = "/kaggle/working"
    MODEL_NAME = "/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1"

TRAIN_PATH      = os.path.join(DATA_DIR, "train.csv")
TEST_PATH       = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

print(f"ENV          : {ENV}")
print(f"MODEL_NAME   : {MODEL_NAME}")
print(f"TRAIN_PATH   : {TRAIN_PATH}")
print(f"SUBMISSION   : {SUBMISSION_PATH}")

ENV          : kaggle
MODEL_NAME   : /kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1
TRAIN_PATH   : /kaggle/input/competitions/lmsys-chatbot-arena/train.csv
SUBMISSION   : /kaggle/working/submission.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import torch

print(f"torch        : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from scipy.special import softmax

RANDOM_SEED = 42
TARGET_COLS = ["winner_model_a", "winner_model_b", "winner_tie"]

torch        : 2.10.0+cu128
CUDA avail   : True
GPU          : Tesla T4


## 2. Load and Inspect Data

In [3]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(f"train : {train.shape}")
print(f"test  : {test.shape}")
print()
print("train columns:", train.columns.tolist())
print("test  columns:", test.columns.tolist())
train.head(2)

train : (57477, 9)
test  : (3, 4)

train columns: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']
test  columns: ['id', 'prompt', 'response_a', 'response_b']


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0


## 3. Preprocessing

In [4]:
# Create integer label from one-hot target columns
# 0 = winner_model_a | 1 = winner_model_b | 2 = winner_tie
train["label"] = np.argmax(
    train[["winner_model_a", "winner_model_b", "winner_tie"]].to_numpy(),
    axis=1
)

print(train[["winner_model_a", "winner_model_b", "winner_tie", "label"]].head())
print()
print(train["label"].value_counts())

   winner_model_a  winner_model_b  winner_tie  label
0               1               0           0      0
1               0               1           0      1
2               0               0           1      2
3               1               0           0      0
4               0               1           0      1

label
0    20064
1    19652
2    17761
Name: count, dtype: int64


In [5]:
def format_input(row):
    return (
        f"[PROMPT]\n{row['prompt']}\n\n"
        f"[RESPONSE A]\n{row['response_a']}\n\n"
        f"[RESPONSE B]\n{row['response_b']}"
    )

train["text"] = train.apply(format_input, axis=1)
test["text"]  = test.apply(format_input, axis=1)

# ── NaN check — NaN inputs propagate through the network and cause NaN loss ───
nan_count = train["text"].isna().sum()
print(f"NaN in train text: {nan_count}")
if nan_count > 0:
    train = train.dropna(subset=["text"])
    print(f"Dropped NaN rows. New train size: {len(train)}")

print("Sample formatted input:")
print(train["text"].iloc[0][:400])

NaN in train text: 0
Sample formatted input:
[PROMPT]
["Is it morally right to try to have a certain percentage of females on managerial positions?","OK, does pineapple belong on a pizza? Relax and give me fun answer."]

[RESPONSE A]
["The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and disc


## 4. Model Training and Validation

In [6]:
train_df, val_df = train_test_split(
    train,
    test_size=0.1,
    random_state=RANDOM_SEED,
    stratify=train["label"]
)

print(f"train_df : {len(train_df):,}")
print(f"val_df   : {len(val_df):,}")

train_df : 51,729
val_df   : 5,748


In [7]:
MAX_LENGTH = 384

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=(ENV == "kaggle"),
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test[["text"]].reset_index(drop=True))

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds   = val_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text"])
val_ds   = val_ds.remove_columns(["text"])
test_ds  = test_ds.remove_columns(["text"])

# Trainer requires the label column to be named "labels"
train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")

train_ds.set_format(type="torch")
val_ds.set_format(type="torch")
test_ds.set_format(type="torch")

print(train_ds)
print(val_ds)
print(test_ds)

The tokenizer you are loading from '/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/51729 [00:00<?, ? examples/s]

Map:   0%|          | 0/5748 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 51729
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5748
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3
})


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    local_files_only=(ENV == "kaggle"),
)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    # ── NaN fixes ────────────────────────────────────────────────────────────
    learning_rate=1e-5,        # lower LR — reduces gradient explosion risk
    weight_decay=0.01,
    max_grad_norm=1.0,         # gradient clipping — caps exploding gradients
    adam_epsilon=1e-6,         # stable epsilon — prevents NaN in Adam updates
    fp16=False,                # disable mixed precision — avoids fp16 overflow
    # ─────────────────────────────────────────────────────────────────────────
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()
print("Training complete.")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                    

Epoch,Training Loss,Validation Loss
1,1.109977,1.097476


Training complete.


In [9]:
val_output = trainer.predict(val_ds)
val_probs  = softmax(val_output.predictions, axis=1)

nan_count = np.isnan(val_probs).sum()
print(f"NaN in val_probs: {nan_count}")
print(f"Sample probs    : {val_probs[:3]}")

val_logloss = log_loss(val_df["label"].values, val_probs)
print(f"Validation log-loss: {val_logloss:.4f}")

NaN in val_probs: 0
Sample probs    : [[0.3596 0.3384 0.3018]
 [0.3596 0.3384 0.3018]
 [0.3596 0.3384 0.3018]]
Validation log-loss: 1.0977


## 5. Inference and Submission

In [10]:
test_output = trainer.predict(test_ds)
test_probs  = softmax(test_output.predictions, axis=1)

submission = pd.DataFrame(test_probs, columns=TARGET_COLS)
submission.insert(0, "id", test["id"].values)

assert submission.shape[0] == len(test),                  "Row count mismatch"
assert not submission.isnull().any().any(),                "Nulls in submission"
assert list(submission.columns) == list(sample_sub.columns), "Column mismatch"

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved to : {SUBMISSION_PATH}")
print(f"Shape    : {submission.shape}")
submission.head()

Saved to : /kaggle/working/submission.csv
Shape    : (3, 4)


/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.359619,0.338379,0.301758
1,211333,0.359619,0.338379,0.301758
2,1233961,0.359619,0.338379,0.301758
